# Basic PMF Example

Demonstrates the core `pmf()` function on synthetic data with known ground truth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from pmf_acls import pmf

rng = np.random.default_rng(42)

# Problem dimensions
n_obs, n_vars, p = 100, 15, 3

# True factors: block-dominant profiles
G_true = np.zeros((p, n_vars))
block = n_vars // p
for k in range(p):
    s, e = k * block, (k + 1) * block if k < p - 1 else n_vars
    G_true[k, s:e] = rng.lognormal(0, 1, size=e - s)
    G_true[k] += rng.lognormal(-2, 0.3, size=n_vars)

F_true = rng.gamma(2, 1, size=(n_obs, p))
X_true = F_true @ G_true

# Add noise
sigma = 0.1 * X_true + 0.01
X_obs = np.maximum(X_true + rng.normal(0, sigma), 0)

In [ ]:
result = pmf(X_obs, sigma, p=3, algorithm="acls", max_iter=500, random_seed=0)

print(f"Converged: {result.converged}")
print(f"Iterations: {result.n_iter}")
print(f"Q: {result.Q:.1f}")
print(f"Explained variance: {result.explained_variance:.4f}")

In [ ]:
# Align recovered profiles to true profiles
def row_norm(M):
    return M / np.maximum(M.sum(axis=1, keepdims=True), 1e-30)

G_true_n = row_norm(G_true)
G_hat_n = row_norm(result.G)

cos_mat = G_true_n @ G_hat_n.T / (
    np.linalg.norm(G_true_n, axis=1, keepdims=True) @
    np.linalg.norm(G_hat_n, axis=1, keepdims=True).T
)
row_ind, col_ind = linear_sum_assignment(1 - cos_mat)

fig, axes = plt.subplots(1, p, figsize=(5 * p, 4), sharey=True)
x = np.arange(n_vars)
for i, ax in enumerate(axes):
    ax.bar(x - 0.15, G_true_n[row_ind[i]], 0.3, label="True", alpha=0.7)
    ax.bar(x + 0.15, G_hat_n[col_ind[i]], 0.3, label="Recovered", alpha=0.7)
    ax.set_title(f"Factor {i+1} (cos={cos_mat[row_ind[i], col_ind[i]]:.3f})")
    ax.set_xlabel("Variable")
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle("True vs Recovered Profiles")
fig.tight_layout()
plt.show()